In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("DSC100_Lab5.ipynb")

# Lab 05: Regex, EDA, and Text Analysis

## Content Warning
This lab includes an analysis of crime in Berkeley. If you feel uncomfortable with this topic, **please contact your TA or the instructors.**

This lab will feature two parts: one that looks at crim and calls to Police Departments and the second text analysis from the New York Times. 

In the first part of this lab, you will be working on visualizing a dataset from the City of Berkeley containing data on calls to the Berkeley Police Department. Information about the dataset can be found [at this link](https://data.cityofberkeley.info/Public-Safety/Berkeley-PD-Calls-for-Service/k2nh-s5h5).

In the second part of this lab, you will analyze New York Times articles discussing trending topics from the past six years.

You will gain experience with:

- Cleaning and exploring a text-based dataset,
- Manipulating data in `pandas` using `string` accessors,
- Creating and interpreting visualizations with `seaborn`,
- Writing and applying regular expressions (regex) with `pandas`, and
- Performing sentiment analysis on text using the `DistilBERT` language model.

To receive credit for a lab, answer all questions correctly and submit before the deadline.
 


## Collaboration Policy

Data science is a collaborative activity. While you may talk with others about this assignment, we ask that you **write your solutions individually**. If you discuss the assignment with others, please **include their names** in the cell below.

**Collaborators:** *list names here*

---
## Debugging Guide
If you run into any technical issues, we highly recommend checking out the [Data 100 Debugging Guide](https://ds100.org/debugging-guide/). In this guide, you can find general questions about Jupyter notebooks / Datahub, Gradescope, and common pandas errors.

### Score Breakdown

 Question | Manual| Points
--- |---| ---
1 |No | 2
2a | No | 2
2b | No | 2
2c | No | 2
2d | Yes | 2
3a | No  | 2
3b | No  | 2
3c | No  | 2
3d | No  | 2
3e | Yes  | 2
4a |No| 1
4b |No| 1
4c |No| 1
5ai |No| 2
5aii |No| 1
5aiii |No| 2
5bi |No| 1
5bii |No| 1
5biii |Yes| 1
5c |No| 2
5d |No| 2
5e |No| 1
5fi |Yes| 1
5fii |Yes| 2
6a |No| 1
6bi |No| 1
6bii |No| 1
6c |Yes| 1
6di |No| 1
6dii |Yes| 1
6e |Yes| 2
**Total** | **8** | **45**

## 🏎️ Before You Start

For each question in the assignment, please write down your answer in the answer cell(s) right below the question.

We understand that it is helpful to have extra cells breaking down the process towards reaching your final answer. If you happen to create new cells below your answer to run code, **NEVER** add cells between a question cell and the answer cell below it. It will cause errors when we run the autograder, and it will sometimes cause a failure to generate the PDF file.

**Important note: The local autograder tests will not be comprehensive. You can pass the automated tests in your notebook but still fail tests on Gradescope after the grades are released.** Please be sure to check your results carefully.

Finally, unless we state otherwise, **do not use for loops or list comprehensions**. The majority of this assignment can be done using built-in commands in `pandas` and `NumPy`.

---
## Setup

In this lab, we'll perform Exploratory Data Analysis and learn some preliminary tips for working with `matplotlib` (a Python plotting library). In the cell below, we configure a custom default figure size. Virtually every default aspect of `matplotlib` [can be customized](https://matplotlib.org/stable/users/explain/customizing.html). 

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 9)

<br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 1: Acquire the Data

Here, we start by acquiring the data that this lab is based on. **Please don't worry about understanding all the code in this section**; the steps shown here are out of scope and provided here for student interest!

**1. Obtain data**<br/>
To retrieve the dataset, we will use the `ds100_utils.download_lab5_data` utility, a helper function. You can see how this is implemented by opening the file `ds100_utils.py` on the left.

In [ ]:
# Run this cell to download the data, no further action is needed.
from ds100_utils import download_lab5_data

dest_path = download_lab5_data()
print(f'Located at {dest_path}')

**2. Unzip file**<br/>
We will now directly unzip the ZIP archive and start working with the uncompressed files.

In [ ]:
# Run this cell to unzip the data, no further action is needed.
my_zip = zipfile.ZipFile(dest_path, 'r')
my_zip.extractall('data')

There is no single right answer regarding whether to work with compressed files in their compressed state or to uncompress them on disk permanently. For example, if you need to work with multiple tools on the same files or write many notebooks to analyze them—and they are not too large—it may be more convenient to uncompress them once. But you may also have situations where you find it preferable to work with the compressed data directly.  

`Python` gives you tools for both approaches, so it can be helpful to know how to perform both tasks in order to choose the one that best suits the problem at hand.

**3. View files**

Now, we'll use the `os` package to list all files in the `data` directory. `os.walk()` recursively traverses the directory, and `os.path.join()` creates the full pathname of each file.

If you're interested in learning more, check out the `Python3` documentation pages for `os.walk` ([link](https://docs.python.org/3/library/os.html#os.walk)) and `os.path.join` ([link](https://docs.python.org/3/library/os.path.html#os.path.join)).

We use `Python3` [format strings](https://docs.python.org/3/tutorial/inputoutput.html) to nicely format the printed variables `dpath` and `fpath`.

In [ ]:
# Run this cell to view the content in the zip file, no further action is needed.
import os

for root, directories, filenames in os.walk('data'):
    # first, print out all directories
    for directory in directories:
        dpath = os.path.join(root, directory)
        print(f"d {dpath}")
        
    # next, print out all files
    for filename in filenames:  
        fpath = os.path.join(root,filename)
        print(f"  {fpath}")

In this Lab, we'll be working with the `Berkeley_PD_-_Calls_for_Service_2022.csv` file. Feel free to check out the other files, though.

<br/>

<hr style="border: 1px solid #fdb515;" />

## Part 2: Clean and Explore the Data

Let's now load the CSV file we have into a `DataFrame`, and start exploring the data. We added a line at the top of the cell to suppress a couple of warnings related to how we use `pd.to_datetime` here, but you need not worry about that.

In [ ]:
%%capture --no-stdout

# Run this cell to read the data into a DataFrame and do some initial formatting, no further action is needed.
calls = pd.read_csv("data/Berkeley_PD_-_Calls_for_Service_2022.csv")
calls['EVENTTM'] = pd.to_datetime(calls['EVENTTM']).dt.strftime('%H:%M:%S %p')

In [ ]:
calls.head()

We see that the fields include a case number, the offense type, the date and time of the offense, the "CVLEGEND" which appears to be related to the offense type, a "CVDOW" which has no apparent meaning, the date the record was added to the database, and the location spread across four fields. We can read more about each field from the City of Berkeley's [open dataset webpage](https://data.cityofberkeley.info/Public-Safety/Berkeley-PD-Calls-for-Service/k2nh-s5h5).

Let's also check some basic information about this `DataFrame` using the `pandas.DataFrame.info` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.info.html)) and `pandas.DataFrame.describe` methods ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html)).

In [ ]:
# df.info() displays name and type of each column, and
# number of non-null entries in each column
calls.info()

Note that the BLKADDR column only has 4476 non-null entries, while the other columns all have 4490 entries. This is because the `.info()` method only counts non-null entries.

In [ ]:
calls.describe()

Notice that the functions above reveal type information for the columns, as well as some basic statistics about the numerical columns found in the `DataFrame`. However, we still need more information about what each column represents. Let's explore the data further in Question 1.

Before we go over the fields to deduce their meanings, the cell below will verify that all the events happened in Berkeley by grouping on the `City` and `State` columns. You should see that all of our data falls into one group.

In [ ]:
# .size() returns the number of rows in each DataFrameGroupBy object
calls.groupby(["City", "State"]).size()

When we called `.head()` on the `DataFrame` `calls`, it seemed like `OFFENSE` and `CVLEGEND` both contained information about the type of event reported. What is the difference in meaning between the two columns? One way to probe this is to look at the `value_counts` for each `Series`.

In [ ]:
calls['OFFENSE'].value_counts().head(10)

In [ ]:
calls['CVLEGEND'].value_counts().head(10)

It seems like `OFFENSE` is more specific than `CVLEGEND`, e.g., "LARCENY" vs. "THEFT FELONY (OVER $950)". If you're unfamiliar with the term, "larceny" is a legal term for theft of personal property.

To get a sense of how many sub-categories there are for each `OFFENSE`, we will set `calls_by_cvlegend_and_offense` equal to a multi-indexed `Series` where the data is first indexed on the `CVLEGEND` and then on the `OFFENSE`, and the data is equal to the number of offenses in the database that match the respective `CVLEGEND` and `OFFENSE`. As you can see, `calls_by_cvlegend_and_offense["LARCENY", "THEFT FROM PERSON"]` returns `13` which means there are 13 instances of larceny with offense of type "THEFT FROM PERSON" in the database.

In [ ]:
calls_by_cvlegend_and_offense = calls.groupby(["CVLEGEND", "OFFENSE"]).size()
calls_by_cvlegend_and_offense["LARCENY", "THEFT FROM PERSON"]

<br>

---

### Question 1

In the cell below, set `answer1` equal to a `list` of strings corresponding to the possible values for `OFFENSE` when `CVLEGEND` is "LARCENY". You can type the answer manually, or you can create an expression that automatically extracts the names.


In [ ]:
answer1 = ...

In [ ]:
grader.check("q1")

<br/>

<hr style="border: 1px solid #fdb515;" />

## Part 3: Visualize the Data


### `Matplotlib` demo

You've seen some `matplotlib` in this class already (in homework 1), but now we will explain how to work with the object-oriented plotting API mentioned in this [matplotlib.pyplot tutorial](https://matplotlib.org/stable/tutorials/pyplot.html). In `matplotlib`, plotting occurs on a set of `Axes` that are associated with a `Figure`. An analogy is that on a blank canvas (`Figure`), you choose a location to plot (`Axes`) and then fill it in (plot).

There are two approaches to labeling and manipulating figure contents, which we'll discuss below. Approach 1 is closest to the plotting paradigm of MATLAB, the namesake of `matplotlib`; Approach 2 is also common because many `matplotlib`-based packages (such as `seaborn`) explicitly return the current set of axes after plotting data. Both are essentially equivalent, and at the end of this class, you'll be comfortable with both. 

**Approach 1**: `matplotlib` will auto-plot onto the current set of `Axes` or (if none exists) create a new figure/set of default axes. You can plot data using methods from `plt`, which is shorthand for the `matplotlib.pyplot` package. Then subsequent `plt` calls all edit the same set of default-created axes.

**Approach 2**:  
After creating the initial plot, you can also use `plt.gca()` to explicitly get the current set of axes and then edit those specific axes using axes methods. Note the method naming is slightly different!

`pandas` also offers basic functionality for plotting. For example, the `DataFrame` and `Series` classes both have a `plot` method, which uses `matplotlib` under the hood. For now, we'll focus on `matplotlib` itself so you get used to the syntax, but just know that convenient `pandas` plotting methods exist for your own future data science exploration.

Below, we show both approaches by generating a horizontal bar plot to visually display the value counts for `CVLEGEND`. See the `barh` [documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.barh.html?highlight=barh#matplotlib.pyplot.barh) for more details.

In [ ]:
# DEMO CELL: assign demo to 1 or 2.
demo = ...

calls_cvlegend = calls['CVLEGEND'].value_counts()

if demo == 1:
    plt.barh(calls_cvlegend.index, calls_cvlegend) # Creates figure and axes
    print(f"Demo {demo}: Using plt methods to update plot")
    plt.ylabel("Crime Category")               # Uses most recently plotted axes
    plt.xlabel("Number of Calls")
    plt.title("Number of Calls by Crime Type")
elif demo == 2:
    print(f"Demo {demo}: Using axes methods to update plot")
    plt.barh(calls_cvlegend.index, calls_cvlegend) # Creates figure and axes
    ax = plt.gca()
    ax.set_ylabel("Crime Category")
    ax.set_xlabel("Number of Calls")
    ax.set_title("Axes methods: Number of Calls by Crime Type")
else:
    print("Error: Please assign the demo variable to 1 or 2.")

plt.show()


### An Additional Note on Plotting in Jupyter Notebooks

You may have noticed that some of our plotting code cells end with a semicolon `;` or `plt.show()`. The former prevents any extra output from the last line of the cell; the latter explicitly returns (and outputs) the figure. Try adding this to your own code in the following questions!

<br>

---

### Question 2

Now it is your turn to make a plot using `matplotlib`. Let's start by transforming the data so that it is easier to work with.

The `CVDOW` field isn't named helpfully, and it is hard to see the meaning from the data alone. According to the website [linked](https://data.cityofberkeley.info/Public-Safety/Berkeley-PD-Calls-for-Service/k2nh-s5h5) at the top of this notebook, `CVDOW` is actually indicating the day that events happened. 0->Sunday, 1->Monday ... 6->Saturday. 

#### Question 2a

Add a new column `Day` to `calls` that has the string weekday (e.g., "Sunday") for the corresponding value in CVDOW. For example, if the first 3 values of `CVDOW` are `[3, 6, 0]`, then the first 3 values of the `Day` column should be `["Wednesday", "Saturday", "Sunday"]`.

**Hint:** Try using the [Series.map](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.map.html) function on `calls["CVDOW"]`. Can you assign this to the new column `calls["Day"]`?

In [ ]:
days = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
day_indices = range(7)
indices_to_days_dict = dict(zip(day_indices, days)) # Should look like {0:"Sunday", 1:"Monday", ..., 6:"Saturday"}

calls["Day"] = ...

In [ ]:
grader.check("q2a")

<br>

---
#### Question 2b

Now let's look at the `EVENTTM` column which indicates the time for events. Since it contains hour and minute information, let's extract the hour info and create a new column named `Hour` in the `calls` `DataFrame`. **You should save the hour as an `int`**. The format of the `'EVENTTM'` column and some of the associated reasoning about the answer in the lab walkthrough is slightly different. However, the answer shown in the walkthrough is still applicable to the question below. 

**Hint:** Your code should only require one line. <br/>

In [ ]:
calls["Hour"] = ...
calls["Hour"]

In [ ]:
grader.check("q2b")

<br>

---
#### Question 2c

Using `matplotlib`, construct a line plot with the count of the number of calls (entries in the table) for each hour of the day  **ordered by the time** (e.g., `12:00 AM`, `1:00 AM`, ...). Be sure that your axes are labeled and that your plot is titled. The solution shown in the lab walkthrough is not the only way to answer this question, an alternative approach could involve using `.sort_index()`.

**Hint 1**: Check out the `plt.plot` method in the `matplotlib` [tutorial](https://matplotlib.org/stable/tutorials/introductory/pyplot.html#intro-to-pyplot), as well as our demo above.

In [ ]:
...

# Leave this for grading purposes.
ax_3d = plt.gca()

In [ ]:
grader.check("q2c")

To better understand the time of day a report occurs, we could **stratify the analysis by the day of the week.** To do this we will use **violin plots** (a variation of a **box plot**).

A violin plot shows an estimated distribution of quantitative data (e.g., distribution of calls by hour) over a categorical variable (day of the week). More calls occur in hours corresponding to the fatter part of each violin; the median hour of all calls in a particular day is marked by the white dot in the corresponding violin.

In [ ]:
# Run this cell to to generate the plot, no further action is needed

import seaborn as sns
ax = sns.violinplot(data=calls.sort_values("CVDOW"),
                    x="Day", y="Hour", hue="Day",
                    saturation=0.5, palette="Set2")
ax.set_title("Stratified Analysis of Phone Calls by Day");

<!-- BEGIN QUESTION -->

<br>

---
#### Question 2d

Based on your line plot and our violin plot above, what observations can you make about the patterns of calls? Here are some dimensions to consider:
* Are there more calls in the day or at night?
* What are the most and least popular times?
* Do call patterns vary by day of the week?


_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

<hr style="border: 1px solid #fdb515;" />

## Part 4: Data Faithfulness vs. Reality
<br>

### Question 3
In this last part of the lab, let's extract the GPS coordinates (latitude, longitude) from the `Block_Location` of each record.

In [ ]:
# An example block location entry.
calls.loc[4, 'Block_Location']

#### Question 3a: Regular Expressions

Use [regular expressions](https://ds100.org/course-notes/regex/regex.html) to create a new `DataFrame` `calls_lat_lon` that has two columns titled `Lat` and `Lon`, containing the respective latitude and longitude of each record in `calls`. You should use the `Block_Location` column to extract the latitude and longitude coordinates.

**Hint**: Check out the `Series.str.extract` [documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.extract.html?highlight=extract#pandas.Series.str.extract).

In [ ]:
calls_lat_lon = ...


calls_lat_lon.head(10)

In [ ]:
grader.check("q3a")

<br>

---

#### Question 3b: Join Tables

Let's include the GPS data into our `calls` data. In the below cell, use `calls_lat_lon` to add two new columns called `Lat` and `Lon` to `calls`.

**Hint 1**: Note that the order of records in `calls` and `calls_lat_lon` are the same. 

**Hint 2**: Another way to achieve our goal could be using `pd.merge`, look through the [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) to see how we can merge using the `left_index` and `right_index` arguments.

In [ ]:
...
calls.sample(5)      # random rows

In [ ]:
grader.check("q3b")

<br>

---
#### Question 3c: Check for Invalid Values

It seems like every record has valid GPS coordinates. That is, there are no `NaN` values in either column as we can verify below

In [ ]:
# Run this cell to obtain fraction of valid lat/lon entries, no further action is needed.
(~calls[["Lat", "Lon"]].isna()).mean()

However, a closer examination of the data reveals something else. Here's the first few records of `calls` again:

In [ ]:
calls.head(5)

There is another field that tells us whether we have a valid `Block_Location` entry per record - i.e., with GPS coordinates (latitude, longitude) that match the listed block location. What is it?

In the below cell, use the field you found to create a new `DataFrame`, `invalid_block_loc`, that contains only the rows of `calls` that have invalid `Block_Location`. Your new `DataFrame` should have all the same columns of `calls`.

In [ ]:
invalid_block_loc = ...
invalid_block_loc.head()

In [ ]:
grader.check("q3c")

<br>

---
#### Question 3d: Patterns in Invalid Values

Now let's explore if there is a pattern to which types of records have invalid block locations.

We've implemented the plotting code for you below, but read through it and verify you understand what we're doing (we've thrown in a bonus `plt.subplots()` call, documentation [here](https://matplotlib.org/stable/gallery/subplots_axes_and_figures/subplots_demo.html#stacking-subplots-in-one-direction)).

In [ ]:
# Run this cell to generate the plot, no further action is needed
missing_by_time = (pd.to_datetime(invalid_block_loc['EVENTDT'], format='%m/%d/%Y %I:%M:%S %p')
                   .value_counts()
                   .sort_index()
                  )
missing_by_crime = (invalid_block_loc['CVLEGEND']
                    .value_counts() 
                    / calls['CVLEGEND'].value_counts()
                   ).dropna().sort_values(ascending=False)

fig, ax = plt.subplots(2)
ax[0].bar(missing_by_time.index, missing_by_time)
ax[0].set_ylabel("Calls with Invalid Data")
ax[1].barh(missing_by_crime.index, missing_by_crime)
ax[1].set_xlabel("Fraction of Invalid Data per Event Type")
fig.suptitle("Characteristics of Invalid Lat/Lon Data")
plt.show()

<!-- BEGIN QUESTION -->


Based on the plots above, are there any patterns among entries that are invalid latitude/longitude data? The dataset information [linked](https://data.cityofberkeley.info/Public-Safety/Berkeley-PD-Calls-for-Service/k2nh-s5h5) at the top of this notebook may also give more context.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---
#### Question 3e: Explore

The below cell plots a map of phone calls by GPS coordinates (latitude, longitude); we drop invalid location data.

In [ ]:
# Run this cell to generate the interactive plot, no further code is needed.
import folium
import folium.plugins

BERKELEY_COORDINATES = (37.87, -122.28)
berkeley_map = folium.Map(location=BERKELEY_COORDINATES, zoom_start=13)
locs = calls.drop(invalid_block_loc.index)[['Lat', 'Lon']].astype('float').values
heatmap = folium.plugins.HeatMap(locs.tolist(), radius=10)
berkeley_map.add_child(heatmap)

<!-- BEGIN QUESTION -->

Based on the above map, what could be some **drawbacks** of using the location fields in this dataset to draw conclusions about crime in Berkeley? This is an open-ended question. Here are some sub-questions to consider:
* Is campus really the safest place to be?
* Why are all the calls located at street intersections, outdoors, and not within buildings?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Lab 03!

**Collaborators**: _list collaborators here_


In this assignment, we will use the [DistilBERT model](https://medium.com/huggingface/distilbert-8cf3380435b5), a Natural Language Processing (NLP) model designed to capture the context and meaning of words within sentences. While you are not expected to understand the intricate details of the model, we will utilize it to perform sentiment analysis on textual data. The necessary tools and the corresponding model are imported below.

- **If you encounter any warnings, please ignore them. As long as the cell executes successfully, there should be no issues.**

## ⚠️ IMPORTANT NOTE

Due to limited computing resources on DataHub, the cell below **may take a significant amount of time to run** (potentially several minutes). This may also apply to other cells later in the assignment that load and use the NLP model.

**Please be patient**, wait, and **avoid restarting the kernel or rerunning these cells** more than necessary. Doing so can slow down *your* notebook and impact *other students* on the same CPU cluster.

Additionally, **DO NOT** open this assignment in multiple tabs or windows. This can cause your notebook to crash and affect DataHub's performance.



In [ ]:
# Run this cell to set up your notebook. 
import warnings
warnings.simplefilter(action="ignore")

import re
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from ds100_utils import *

# Ensure that pandas shows at least 280 characters in columns, so we can see full articles.
pd.set_option("max_colwidth", 280)
plt.style.use("fivethirtyeight")
sns.set()
sns.set_context("talk")
sns.set_palette("colorblind")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Question 4: Importing the Data

The data for this assignment is sourced from the [New York Times (NYT) Archive API](https://developer.nytimes.com/docs/archive-product/1/overview), which provides information about articles published in the past.

The file `data/nyt_articles.txt` contains filtered data of specific NYT articles published between 2019 and 2024 (inclusive). Each article discusses trending topics, which we will specify shortly.

<br>

---

### Question 4a

Let's examine the contents of the `data/nyt_articles.txt` file.

Using the [`open()` function](https://docs.python.org/3/library/functions.html#open) and the [`read()` method](https://docs.python.org/3/tutorial/inputoutput.html#methods-of-file-objects) of a `python` file object, read **the first 330 characters** of the file `data/nyt_articles.txt` and store the result in the variable `q1a`. Then, print the result to inspect it.

**CAUTION:** Viewing the contents of large files in a Jupyter Notebook can crash your browser. Be careful not to print the entire contents of the file.

In [ ]:
...
print(q1a)

In [ ]:
grader.check("q4a")

<br>

---

###  Question 4b

Based on the printed output from `q4a`, what format is the data in?

**A.** CSV<br/>
**B.** JavaScript Object Notation (JSON)<br/>
**C.** HTML<br/>
**D.** Excel (XLSX)

Answer in the following cell. Your answer should be a string, either `"A"`, `"B"`, `"C"`, or `"D"`, stored in the variable `q1b`.

**CAUTION:** Viewing the contents of large files in a Jupyter Notebook can crash your browser. Be careful not to print the entire contents of the file, and do not use the file explorer to open data files directly.

In [ ]:
q1b = ...

In [ ]:
grader.check("q4b")

<br>

---

###  Question 4c
`pandas` has built-in readers for many different file formats. To learn more about these, check out the documentation:

- `pd.read_csv` [(docs)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)
- `pd.read_json` [(docs)](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html)
- `pd.read_html` [(docs)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_html.html)
- `pd.read_excel` [(docs)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_excel.html)

Load the file `data/nyt_articles.txt` as a `DataFrame`, and store it in the variable `news_df`.

**Hint:** If your code is taking a while to run, you should review your answers to `q4a` and `q4b`; you may have used the incorrect data loading function.

In [ ]:
...
news_df.head()

In [ ]:
grader.check("q4c")

<br/>

<hr style="border: 1px solid #fdb515;" />

##  Question 5: Topic Trends Over Time

Now that we've loaded the NYT data, let's analyze trends in different topics. This will help us understand how various subjects have evolved over the years and identify any significant patterns or shifts in public interest.

We will start by extracting date and time information from the articles and then proceed to analyze the frequency and context of specific topics mentioned in the articles.


<br>


---

###  Question 5a

In this question, we will process the `pub_date` column of our dataset to canonicalize time-related data.
This will help us investigate the trend of news articles across units of time like years, months, and seasons.

####  Question 5a, Part i

Create a new `DataFrame` called `dates` that contains:
1. The same index as `news_df`
2. Three columns: `Month`, `Year`, and `Minute`, which contain the month, year, and minute, respectively, that each article was published.

Additionally, convert all numerical values (`Month`, `Year`, `Minute`) to type `int`.

**Note:** For this problem, you are not permitted to use methods from the `Series.dt` accessor.

**Hint 1:** Use the `Series.str.extract` function ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html)).

**Hint 2:** Use raw strings and capture groups. You may find it helpful to copy the example date and time entries above into [regex101.com](https://regex101.com/) to experiment with regular expressions.

**Hint 3:** It might be helpful to break this up into a couple of steps (e.g., first extract date values and then extract time values).

In [ ]:
...

In [ ]:
grader.check("q5ai")

---
####  Question 5a, Part ii

We aim to analyze topic trends over time by merging news article data with corresponding date and time data. 

Use the `pd.DataFrame.merge` [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) to merge the `dates` `DataFrame` with the `news_df` `DataFrame`. Ensure that `news_df` is the left `DataFrame` and `dates` is the right `DataFrame` in the merge operation.

Assign the merged `DataFrame` to a variable named `news_df_dates`.

In [ ]:
news_df_dates = ...

In [ ]:
grader.check("q5aii")

#### Question 5a, Part iii

Add a column to `news_df_dates` called `Quarter` that contains the [fiscal quarter](https://www.investopedia.com/terms/q/quarter.asp#:~:text=The%20standard%20calendar%20quarters%20that,August%2C%20and%20September%20(Q3)) each news article was published.

Each value of `Quarter` should be in the format `"<Year>Q<Quarter Number>"`.

For example:
- A news article published in May 2021 will have its `Quarter` value as `"2021Q2"`.
- A news article published in October 2023 will have its `Quarter` value as `"2023Q4"`.

Do not hardcode the conversion from month to quarter (e.g., using the dictionary `{1: 'Q1', 2: 'Q1', ..., 12: 'Q4'}`). Instead, perform a mathematical operation to convert the month to quarter.

**Hint:** Adding two `Series` of strings (e.g., `ser_1 + ser_2`) performs an element-wise concatenation of their elements.

In [ ]:
...

In [ ]:
grader.check("q5aiii")

<br>

---

###  Question 5b

In this question, we will answer some EDA questions about `news_df_dates`.

####  Question 5b, Part i
In `news_df_dates`, suppose we create a new column `num_google_mentions` that records the number of times the word `"google"` is mentioned in each news article. What type of variable is `num_google_mentions`?

**A.** Quantitative variable<br/>
**B.** Qualitative Ordinal variable<br/>
**C.** Qualitative Nominal variable

Answer in the following cell. Your answer should be a string, either `"A"`, `"B"`, or `"C"`, stored in the variable `q5bi`.

In [ ]:
q5bi = ...

In [ ]:
grader.check("q5bi")

####  Question 5b, Part ii
Which of the following options best describes the granularity of `news_df_dates`? 

Each row in `news_df_dates` uniquely describes:

**A.** A calendar date. <br/>
**B.** An hour of a calendar date. <br/>
**C.** A news article.

Answer in the following cell. Your answer should be a string, either `"A"`, `"B"`, or `"C"`, stored in the variable `q5bii`.

In [ ]:
q5bii = ...

In [ ]:
grader.check("q5bii")

<!-- BEGIN QUESTION -->

####  Question 5b, Part iii

Suppose we wanted to investigate trends in how often the word `"AI"` is mentioned in NYT articles since the 1980s.

Is `news_df` a suitable dataset for this investigation? Explain your reasoning.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---

###  Question 5c

Some news articles include quotes in their lead paragraph (i.e., first paragraph) to grab the reader's attention and provide additional context. For the purposes of this question, a quote is defined as a sequence of characters starting with the character `"` and ending with a period (`.`), question mark (`?`), or exclamation point (`!`), followed by a closing `"`. For example:
- `"The mitochondria is the powerhouse of the cell!"`
- `"Did DATA C100 course staff host a social event with staff from DATA C8?"`
- `"R is great." A TA said, "but have you tried using Python?"`

If we follow the definition above, the following text snippet contains two quotes:
- `The TA asked, "What's the purpose of regular expressions?" The student thought for a moment and then replied, "Regex are used to identify patterns in text."`



Brandon wants to extract individual quotes from paragraphs using the definition of a quote given above.
Brandon proposes the regex pattern `r'\".*[\.\?\!]\"'`. Unfortunately, Brandon's pattern identifies only one quote in the test string below instead of two.

Modify Brandon's regex pattern so that it correctly matches the two quotes individually. Store your new pattern in the variable `modified_pattern`.

In [ ]:
test_string = '"How are your classes?" the student asked. "Super challenging! But a lot of fun!" said their roommate.'

print("Test string:", test_string)
print("Original pattern identifies:", re.findall(r'\".*[\.\?\!]\"', test_string))

modified_pattern = ...
print("Modified pattern identifies:", re.findall(modified_pattern, test_string))

In [ ]:
grader.check("q5c")

<br>

---

###  Question 5d

Next, we will investigate popularity trends of the following four topics: [New Year](https://en.wikipedia.org/wiki/New_Year), [Wordle](https://www.nytimes.com/games/wordle/index.html), [Zoom](https://www.zoom.com/), and [GPT models](https://en.wikipedia.org/wiki/Generative_pre-trained_transformer).

For each topic, add an integer column to `news_df_dates` indicating the number of times the topic is mentioned in the `lead_paragraph` (i.e., first paragraph) of each article. 

- The columns should be named `"New Year"`,  `"Zoom"`, `"Wordle"`, and `"GPT Model"`.

- You may use a `for` loop to iterate over a list of the four topics.

Here are the definitions of a single "mention" of each topic:
- New Year: An appearance of `"New Year"` or `"New Years"`, surrounded by non-word characters, in the lead paragraph of the article. For example, `"Happy New Year!"` is a match.
- Wordle: `"Wordle"`, surrounded by non-word characters. For example, `"Wordless"` would not match.
- Zoom: `"Zoom"`, surrounded by non-word characters. For example, `"Zoomer"` would not match.
- GPT Model: either (1) a consecutive sequence of alphabetical characters, followed by an optional dash (`-`), then `GPT`; or (2) `GPT`, then a dash (`-`), then a numeric digit, then an optional alphabetical character.
    - For example, these words match: `"ChatGPT"`, `"CHAT-GPT"`, `"GPT-3"`, `"GPT-4o"`.
    - However, these words do not match: `"chatgpt"`, `"chatgpt-4o"`.


In [ ]:
...

news_df_dates.head(1)

In [ ]:
grader.check("q5d")

<br>

---

###  Question 5e

Create a new `DataFrame` called `topic_mentions` with the following characteristics:

- There should be one column for each topic (`"New Year"`,  `"Zoom"`, `"Wordle"`, and `"GPT Model"`).

- The index should be `Quarter`.

- The values are the number of articles that mentioned each topic in each quarter. 

**Hint**: Define a helper function `num_mentioned(ser)`, which takes a `Series` object `ser` and returns the number of entries in `ser` that are larger than `0`.

In [ ]:
topics = ["New Year", "Wordle", "Zoom", "GPT Model"]

...

# Year 2023 records
topic_mentions[16:20]

In [ ]:
grader.check("q5e")

<!-- BEGIN QUESTION -->

<br>

---
### Question 5f

Let's visualize the article counts for each topic by quarter from 2019 to 2024.

#### Question 5f, Part i

Using `sns.lineplot` ([documentation](https://seaborn.pydata.org/generated/seaborn.lineplot.html)) and `topic_mentions`, visualize the topic trends across quarters. Your plot should look like this:

<center>
    <img src="./images/topic_mentions.png" width="750" align="left" alt="topic mentions 2019 to 2024">
</center>



In [ ]:
plt.figure(figsize=(12, 5)) # DO NOT MODIFY

for topic in topics:
    ...

# DO NOT MODIFY THE CODE BELOW
# If your solution above is correct, running this cell should produce the plot above.
plt.xticks(rotation=60)
plt.yticks()
plt.ylabel("Number of Articles")
plt.xlabel("Quarter")
plt.title("Number of Articles Released (2019-2024)")
plt.gcf().set_facecolor('white')
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

####  Question 5f, Part ii

For each of the four topics, identify one interesting pattern in the visualization and provide a tentative explanation of why you think the pattern exists.


<br>

<center>
    <img src = "images/topic_mentions.png" width = "750" align="left" alt="Topic mentions 2019 to 2024">
</center>

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

<hr style="border: 1px solid #fdb515;" />

##  Question 6: Sentiment Analysis

**Sentiment analysis** involves using an NLP model to classify the emotional tone of text. For example, "You're great!" has a positive sentiment, while "I feel horrible" has a negative sentiment.

In this section, we will explore temporal changes in the **sentiment** of NYT articles that mention each topic.

> The sentiment values in this section were computed using a fine-tuned version of the **DistilBERT** model ([GitHub](https://github.com/huggingface/transformers/tree/main/examples/research_projects/distillation), [original paper](https://arxiv.org/abs/1910.01108)).
>
> DistilBERT is a neural network-based language model similar to ChatGPT. These models are not covered in DSC 100, and we don't expect you to know how they work. If you're interested in learning more, consider taking Machine Learning courses.
>
> The [HuggingFace library](https://huggingface.co/) was used to build the sentiment analysis pipeline and load the model. [Here](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) is the **model card** of the `DistilBERT` model we used. The model card contains general information about the model, such as training arguments and training data. Again, you don't need to know these details for Data 100!

Run the following three cells to set up the sentiment analysis pipeline and see examples of how we can get the sentiment for different strings.


In [ ]:
from transformers import pipeline
model_checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

In [ ]:
# Load the model
sentiment_analysis = pipeline("sentiment-analysis", model=model_checkpoint)

In [ ]:
# Get the sentiment of a given string
sentiment_1 = sentiment_analysis("I have two dogs.")
print("Example 1: " + str(sentiment_1))

sentiment_2 = sentiment_analysis("I do not have dogs.")
print("Example 2: " + str(sentiment_2))

sentiment_3 = sentiment_analysis("Fortunately, I do not have dogs to worry about.")
print("Example 3: " + str(sentiment_3))

As you can see, the model can determine the sentiment of phrases/sentences (not just words). The model measures the phrase's **polarity**, indicating how strongly negative or positive it is on a scale of 0 to 1.

**Note:** The output is a list, and each list element is a dictionary with two keys (label and score). Note that we could have gotten the sentiments of the two sentences by putting them in a list (batch) and then running the pipeline once (see the code below).


In [ ]:
sentiments = sentiment_analysis(["I have two dogs.", "I do not have dogs."])
print(sentiments)

<br>

---
###  Question 6a

Try it out yourself! The sentences we provided in the previous example have pretty high polarity scores. Let's see how the model behaves with more ambiguous sentences.

Write a sentence `less_polar_sentence` that has a polarity score of less than 0.8. This may take some trial and error. Let this be an opportunity to think about whether the model works as you'd expect.

In [ ]:
less_polar_sentence = ...
results = sentiment_analysis(less_polar_sentence)
print(results)

In [ ]:
grader.check("q6a")

<br>

---

###  Question 6b

As a first step, let's obtain the sentiment of the NYT articles that mention these **three** topics: `Zoom`, `New Year`, and `GPT`. 

**Note:** We will not analyze the sentiment of articles that mention `Wordle`.

####  Question 6b, Part i

Create a `DataFrame` called `news_df_filtered` that contains all articles from `news_df_dates` that mention `Zoom`, `New Year`, or `GPT`, but do not mention `Wordle`. Use the same definitions from Question 5c.

In [ ]:
news_df_filtered = ...

In [ ]:
grader.check("q6bi")

####  Question 6b, Part ii

Having everyon run the `DistilBERT` model for all articles is too computationally intensive for JupyterHub. So, we have done this part for you and saved the article sentiment scores in  `nyt_sentiments.csv`:

1. Load the file `data/nyt_sentiments.csv` as a `DataFrame` called `sentiment`. 
2. Set the index of `sentiment` as the `web_url` of each article, which uniquely defines each article.

Note that the model outputs both a label and a score. After importing the data, we recommend taking a look at its structure before attempting the next part.

For each article, `sentiment` provides an outputted `label` and 0 to 1 sentiment `score`. Scores closer to 1 have a stronger sentiment, while scores closer to 0 are more neutral. We can more efficiently communicate the information in these two columns with a single new column. 

3. Add a new column called `article_sentiment` to `sentiment` that provides the existing score if the label is "POSITIVE", and negates the existing score if the label is "NEGATIVE". For example, a score of `0.6` with a label `POSITIVE` would have the value `0.6` in `article_sentiment`, while a score of `0.6` with a label `NEGATIVE` would have the value `-0.6` in `article_sentiment`.

4. Merge `news_df` with `sentiment` using the `web_url` column, making sure that `news_df` is the left `DataFrame` while `sentiment` is the right `DataFrame`. The merged `DataFrame` should be called `news_df_sentiment`.

5. Finally, drop the original `label` and `score` columns.

In [ ]:
...

In [ ]:
grader.check("q6bii")

<!-- BEGIN QUESTION -->

<br>

---
###  Question 6c

Let's now visualize the distribution of article sentiment.

Using `seaborn`, we created a histogram to visualize the distribution of `article_sentiment`. Run the cell below to display the plot.

In [ ]:
sns.histplot(data=news_df_sentiment, x='article_sentiment')
plt.xlabel('Sentiment of Leading Paragraphs')
plt.title('Histogram of Leading Paragraph Sentiment')
plt.plot();

Are you at all surprised by the distribution of sentiment in the graph above? Describe what you notice about the graph and how it relates to what you learned in part **6a**.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---
###  Question 6d

Let's audit our data to better understand how well the sentiment analysis model works with our specific dataset. It's good practice to compare our assumptions to model outputs.

####  Question 6d, Part i

Assign `top_positive` and `top_negative` to `DataFrame`s containing the five articles with the highest `article_sentiment` values and the five lowest `article_sentiment` values, respectively. The `DataFrame`s should have the columns `lead_paragraph` and `article_sentiment`.

In [ ]:
top_positive = ...
top_negative = ...

display(top_positive, top_negative)

In [ ]:
grader.check("q6di")

<!-- BEGIN QUESTION -->

####  Question 6d, Part ii

Do you agree with the current sentiment-based ordering of news articles, or would you rearrange the ordering? Do you feel that the DistilBERT model is a good model for our task of analyzing sentiment in news articles?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---

### Continued Visualizing
Let's continue to visualize the `news_df_sentiment` data. The cell below adds a new datetime column `date` to `news_df_sentiment`. The datetime format will make visualization easier.

In [ ]:
# Combine the columns into a single date string in 'YYYY-MM-DD' format
news_df_sentiment['date_str'] = (
    news_df_sentiment['Year'].astype(str)
    + '-' + news_df_sentiment['Month'].astype(str)
    + '-' + news_df_sentiment['pub_date'].str[8:10]
)

# Convert the combined string to a datetime object using pd.to_datetime()
news_df_sentiment['date'] = pd.to_datetime(news_df_sentiment['date_str'], format='%Y-%m-%d', errors='coerce')

Below, we visualize the change in sentiment in the topic `Zoom` over time, using `sns.lineplot` to plot `date` on the x-axis and `article_sentiment` on the y-axis.

**Note**: If the following plot is empty, please rerun from all cells starting from Part 3b where `news_df_sentiment` was initialized.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=news_df_sentiment[news_df_sentiment["Zoom"] > 0], x='date', y='article_sentiment')
plt.xticks(rotation=70);

**This plot is not very pretty!** This isn't because of any errors on your part. Instead, we need to use a different visualization method to understand our data.

<!-- BEGIN QUESTION -->

###  Question 6e

Let's visualize our data more effectively. We will still use `sns.lineplot`, but instead of plotting every observation, we will first aggregate our data, and then plot the aggregated values.

We will also compare sentiment scores across three topics: `New Year`, `Zoom`, and `GPT`.

We will use the `DataFrame` `news_df_sentiment` in this question.

1. For each topic, generate a `DataFrame` that shows the average article sentiment for each quarter. In each `DataFrame`, be sure to include a column called `Topic` that has the same string value in every row (either `New Year`, `Zoom` or `GPT`).
2. Concatenate the `DataFrame`s obtained from step (1) using `pd.concat` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)). Assign this to `all_topic_qtr_avg_sentiments`.
3. Finally, we have provided the code to plot each topic's average article sentiment in each quarter using `all_topic_qrt_avg_sentiments`.

Your graph should have a similar title, axis labels, markers, and x-axis tick label ordering as the one below.

<img src = "images/sentiment_graph.png" width = "800">

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
dfs_per_topic = []

for topic in topics:
    df_of_current_topic = ...
    df_of_current_topic["Topic"] = ...
    ...

all_topic_qtr_avg_sentiments = ...
sns.lineplot(data=all_topic_qtr_avg_sentiments, x="Quarter", y="article_sentiment", hue="Topic")

plt.title('Avg. Sentiment per Topic Across Quarters')
plt.xlabel('Time')
plt.ylabel('Lead Paragraph Sentiment')

# If the above are implemented correctly, running this cell should produce the graph shown above.
plt.axhline(0, color='black')
plt.xticks(rotation=65);

<!-- END QUESTION -->

<br>

---

### Takeaways

In this lab, we used a language model to evaluate the sentiment of news articles and quantify text data (qualitative data) so that we could perform data analysis on a large set of journalism data. Though we used the [HuggingFace DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) model, there are thousands of language models available for use, and with rapid innovations in the NLP research space, there are new models frequently being created. 

Different models evaluate sentiment differently. You may have noticed that the DistilBERT model struggles with evaluating neutral sentences and often gives sentences a high polarity score. When evaluating which models to use in your projects, it's useful to test them on small inputs of data to see how they perform, like we did by testing out various sentences! Different models may perform differently (often due to how the model was trained and created), so it's important to understand these differences when deciding what model to use for your data.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Lab 5!


### Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Once you submit this file to the HW 3 Coding assignment on Gradescope, Gradescope will automatically submit a PDF file with your written answers to the Lab 5 Written assignment. 

**Important**: Please check that your written responses were generated and submit it correctly to the Lab 5 Written Assignment.

**You are responsible for ensuring your submission follows our requirements and that the PDF for Lab 4 written answers was generated/submitted correctly. We will not be granting regrade requests nor extensions to submissions that don't follow instructions.** If you encounter any difficulties with submission, please don't hesitate to reach out prior to the deadline.


## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)